# NB10 — Polynomial Regression and Interaction Terms

> **Linear regression is linear in *parameters*, not in *variables*.**
> `y = β₀ + β₁x + β₂x²` is still linear regression — x² is just another feature.


## Polynomial regression

Add powers of x as new features:

```
y = β₀ + β₁x + β₂x² + β₃x³ + ε
```

This is still OLS — we just expand the design matrix with x², x³, etc.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score

np.random.seed(0)
X = np.sort(np.random.uniform(-3, 3, 80))
y = 0.5*X**3 - X**2 + 2*X + np.random.normal(0, 3, 80)

X_2d = X.reshape(-1, 1)
X_plot = np.linspace(-3, 3, 200).reshape(-1, 1)

fig, axes = plt.subplots(1, 4, figsize=(17, 4))
for ax, deg in zip(axes, [1, 2, 3, 8]):
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
        ('ols',  LinearRegression()),
    ])
    pipe.fit(X_2d, y)
    y_plot = pipe.predict(X_plot)
    r2 = pipe.score(X_2d, y)

    ax.scatter(X, y, s=15, alpha=0.5, color='steelblue')
    ax.plot(X_plot, y_plot, 'r-', linewidth=2)
    ax.set_title(f'degree={deg}\nR²={r2:.3f}')
    ax.set_ylim(-25, 30); ax.grid(alpha=0.3)

plt.suptitle('Polynomial regression — degree controls flexibility', fontsize=12)
plt.tight_layout(); plt.show()


## Underfitting vs Overfitting

| Degree | Situation | Symptom |
|--------|-----------|---------|
| Too low (1) | Underfitting | High train error, high test error |
| Just right (3) | Good fit | Low train error, low test error |
| Too high (8+) | Overfitting | Low train error, HIGH test error |

Always use cross-validation to choose the degree.


In [ ]:
# Train/test split to see overfitting
from sklearn.model_selection import train_test_split, cross_val_score
import numpy as np

X_2d = X.reshape(-1, 1)
X_tr, X_te, y_tr, y_te = train_test_split(X_2d, y, test_size=0.3, random_state=42)

degrees = range(1, 12)
train_r2, test_r2, cv_r2 = [], [], []

for deg in degrees:
    pipe = Pipeline([('poly', PolynomialFeatures(deg)), ('ols', LinearRegression())])
    pipe.fit(X_tr, y_tr)
    train_r2.append(pipe.score(X_tr, y_tr))
    test_r2.append(pipe.score(X_te, y_te))
    cv = cross_val_score(pipe, X_2d, y, cv=5, scoring='r2')
    cv_r2.append(cv.mean())

import matplotlib.pyplot as plt
plt.figure(figsize=(8, 4))
plt.plot(degrees, train_r2, 'o-', label='Train R²', color='steelblue')
plt.plot(degrees, test_r2,  's-', label='Test R²',  color='crimson')
plt.plot(degrees, cv_r2,    '^--',label='5-fold CV R²', color='green')
plt.axvline(3, color='gray', linewidth=1, linestyle='--', label='True degree=3')
plt.xlabel('Polynomial degree'); plt.ylabel('R²')
plt.title('Underfitting → Good fit → Overfitting')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


## Interaction terms

An interaction term captures **when the effect of x₁ on y depends on x₂**:

```
y = β₀ + β₁x₁ + β₂x₂ + β₃(x₁·x₂) + ε
```

Interpretation of β₃:
> "How much does the slope of x₁ change for each unit increase in x₂?"


In [ ]:
import numpy as np
import statsmodels.api as sm

np.random.seed(7)
n = 200
# Two groups: low education (edu=0) and high education (edu=1)
edu   = np.random.binomial(1, 0.5, n)
exper = np.random.uniform(1, 20, n)
# Salary: experience matters MORE for high-education workers
salary = 30 + 2*exper + 15*edu + 3*exper*edu + np.random.normal(0, 5, n)

# Model WITHOUT interaction
X_no  = sm.add_constant(np.column_stack([exper, edu]))
res_no = sm.OLS(salary, X_no).fit()

# Model WITH interaction
X_int  = sm.add_constant(np.column_stack([exper, edu, exper*edu]))
res_int = sm.OLS(salary, X_int).fit()

print("=== Without interaction ===")
print(res_no.summary().tables[1])
print("\n=== With interaction term (exper × edu) ===")
print(res_int.summary().tables[1])

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 5))
for g, color, label in [(0,'steelblue','Low education'), (1,'crimson','High education')]:
    mask = edu == g
    ax.scatter(exper[mask], salary[mask], s=15, alpha=0.4, color=color)
    xs = np.linspace(1, 20, 100)
    # Fitted line from interaction model
    b = res_int.params
    ys = b[0] + b[1]*xs + b[2]*g + b[3]*xs*g
    ax.plot(xs, ys, color=color, linewidth=2.5, label=label)

ax.set_xlabel('Experience'); ax.set_ylabel('Salary')
ax.set_title('Interaction: slope of experience differs by education group')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
print(f"\nWithout interaction: same slope for both groups")
print(f"With interaction: slope for high-edu = {res_int.params[1]+res_int.params[3]:.2f}")
print(f"With interaction: slope for low-edu  = {res_int.params[1]:.2f}")


## Key Takeaways

| Concept | Key point |
|---------|-----------|
| Polynomial | Add x², x³ as new columns — still OLS |
| Degree selection | Use cross-validation, not just train R² |
| Interaction | β₃(x₁x₂) — effect of x₁ depends on x₂ |
| Interpretation | Marginal effect of x₁ = β₁ + β₃x₂ |

**Next:** NB11 — Ridge Regression (L2 regularisation)
